# EcoCrop GEE - 작물 기후 적합성 평가

Google Earth Engine을 활용한 전 세계 작물 적합성 분석 도구

## 지원하는 데이터 소스
| 소스 | 시계열 | 해상도 | 토양수분 | 특징 |
|------|--------|--------|----------|------|
| WorldClim | 1970-2000 평균 | 1km | X | 빠른 기후 평가 |
| ERA5 | 1950-현재 | 11km | X | 실시간 기후 |
| TerraClimate | 1958-현재 | 4km | X | 고해상도 |
| **FLDAS** | **1982-현재** | **11km** | **O** | **토양수분 포함!** |

## 1. 설치 및 초기화

In [ ]:
# 필요한 패키지 설치 (최초 1회)
# !pip install earthengine-api pandas numpy matplotlib folium geemap

In [ ]:
# 모듈 임포트
from ecocrop_gee import (
    EcoCropGEE,
    create_custom_crop_parameters,
    load_ecocrop_database,
    get_crop_parameters,
    get_soil_moisture_preset,
    SOIL_MOISTURE_PRESETS
)
import pandas as pd
import matplotlib.pyplot as plt

## 2. 기본 분석 (WorldClim 사용)

In [ ]:
# 기본 분석 예제 - 밀 적합성 (한국)
analyzer = EcoCropGEE()
analyzer.load_database('EcoCrop_DB_secondtrim.csv')
analyzer.set_crop('wheat')
analyzer.set_roi_point(lon=127.0, lat=37.5, buffer_km=100)
analyzer.fetch_climate(source='worldclim')
analyzer.calculate_suitability(method='annual')
stats = analyzer.get_statistics()

---
# FLDAS를 활용한 고급 분석

FLDAS (Famine Early Warning Systems Network Land Data Assimilation System)는 **토양 수분**까지 포함하여 더 정밀한 작물 적합성 평가가 가능합니다.

## 3. FLDAS 기본 분석 (토양 수분 포함)

In [ ]:
# FLDAS를 사용한 분석 - 토양수분까지 포함!
analyzer = EcoCropGEE()
analyzer.load_database('EcoCrop_DB_secondtrim.csv')

# 작물 선택 및 토양수분 파라미터 추가
analyzer.set_crop('rice')

# 토양수분 프리셋 적용 (선택사항)
analyzer.crop_params.update(get_soil_moisture_preset('rice'))
print("토양수분 파라미터:", {k: v for k, v in analyzer.crop_params.items() if 'SM' in k})

In [ ]:
# 지역 및 기간 설정
analyzer.set_roi_point(lon=127.0, lat=35.0, buffer_km=50)  # 전라남도

# FLDAS 데이터 가져오기 (2020년)
analyzer.fetch_climate(
    start_date='2020-01-01',
    end_date='2020-12-31',
    source='fldas'  # FLDAS 사용!
)

In [ ]:
# 적합성 계산 (토양수분 포함)
# 방법 1: 제한 요인 방식 (기본) - 가장 낮은 점수가 최종 점수
analyzer.calculate_suitability(method='annual')

# 방법 2: 가중 평균 방식
# analyzer.calculate_suitability(
#     method='annual',
#     weights={'temp': 0.4, 'precip': 0.3, 'soil_moisture': 0.3}
# )

stats = analyzer.get_statistics()

In [ ]:
# 지도 시각화
m = analyzer.create_map()
m

## 4. FLDAS 월별 시계열 분석

In [ ]:
# 월별 데이터로 시계열 분석
analyzer2 = EcoCropGEE()
analyzer2.load_database('EcoCrop_DB_secondtrim.csv')
analyzer2.set_crop('rice')
analyzer2.crop_params.update(get_soil_moisture_preset('rice'))
analyzer2.set_roi_point(lon=127.0, lat=35.0, buffer_km=50)

# FLDAS 월별 데이터 가져오기
analyzer2.fetch_fldas_monthly(
    start_date='2020-01-01',
    end_date='2020-12-31'
)

In [ ]:
# 월별 적합성 점수 계산
analyzer2.calculate_monthly_scores(
    weights={'temp': 0.4, 'precip': 0.3, 'soil_moisture': 0.3}
)

# 월별 시계열 데이터 추출
monthly_df = analyzer2.get_monthly_timeseries()
monthly_df

In [ ]:
# 월별 적합성 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 종합 점수
axes[0, 0].bar(monthly_df.index, monthly_df['overall_score'], color='steelblue')
axes[0, 0].set_title('월별 종합 적합성 점수')
axes[0, 0].set_ylabel('점수')
axes[0, 0].set_ylim(0, 100)
axes[0, 0].tick_params(axis='x', rotation=45)

# 온도 점수
axes[0, 1].bar(monthly_df.index, monthly_df['temp_score'], color='coral')
axes[0, 1].set_title('월별 온도 적합성')
axes[0, 1].set_ylabel('점수')
axes[0, 1].set_ylim(0, 100)
axes[0, 1].tick_params(axis='x', rotation=45)

# 강수량 점수
axes[1, 0].bar(monthly_df.index, monthly_df['precip_score'], color='teal')
axes[1, 0].set_title('월별 강수량 적합성')
axes[1, 0].set_ylabel('점수')
axes[1, 0].set_ylim(0, 100)
axes[1, 0].tick_params(axis='x', rotation=45)

# 토양 수분 점수
axes[1, 1].bar(monthly_df.index, monthly_df['soil_moisture_score'], color='saddlebrown')
axes[1, 1].set_title('월별 토양 수분 적합성')
axes[1, 1].set_ylabel('점수')
axes[1, 1].set_ylim(0, 100)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. 재배 시즌 분석

In [ ]:
# 벼 재배 시즌 (5월-10월) 적합성 계산
analyzer2.calculate_seasonal_score(
    start_month=5,   # 5월
    end_month=10,    # 10월
    aggregation='mean'  # 'mean', 'min', 또는 'product'
)

# 시즌 통계
from ecocrop_gee import get_regional_statistics
seasonal_stats = get_regional_statistics(
    analyzer2.seasonal_scores,
    analyzer2.roi,
    scale=1000
)

print("\n재배 시즌 (5-10월) 적합성:")
for k, v in seasonal_stats.items():
    if v is not None:
        print(f"  {k}: {v:.2f}")

## 6. 사용자 정의 작물 (토양 수분 포함)

In [ ]:
# 토양 수분 프리셋 확인
print("사용 가능한 토양 수분 프리셋:")
for crop, params in SOIL_MOISTURE_PRESETS.items():
    print(f"  {crop}: {params}")

In [ ]:
# 사용자 정의 작물 - 토양 수분 파라미터 포함
custom_soybean = create_custom_crop_parameters(
    name="한국 콩",
    # 온도 파라미터 (°C)
    topmn=20, topmx=28,    # 최적 온도 범위
    tmin=10, tmax=38,      # 허용 온도 범위
    # 강수량 파라미터 (mm/년)
    ropmn=600, ropmx=1200, # 최적 강수량
    rmin=400, rmax=2000,   # 허용 강수량
    # 생육 기간 (일)
    gmin=100, gmax=150,
    # 토양 수분 파라미터 (m³/m³)
    sm_min=0.15,           # 위조점 (너무 건조)
    sm_opt1=0.25,          # 최적 하한
    sm_opt2=0.38,          # 최적 상한
    sm_max=0.45            # 과습점 (침수)
)

print("사용자 정의 콩 파라미터:")
for k, v in custom_soybean.items():
    print(f"  {k}: {v}")

In [ ]:
# 사용자 정의 작물로 분석
analyzer3 = EcoCropGEE()
analyzer3.set_crop(custom_params=custom_soybean)
analyzer3.set_roi_point(lon=127.5, lat=36.5, buffer_km=50)  # 충청도
analyzer3.fetch_climate(
    start_date='2020-01-01',
    end_date='2020-12-31',
    source='fldas'
)
analyzer3.calculate_suitability(
    method='annual',
    weights={'temp': 0.35, 'precip': 0.30, 'soil_moisture': 0.35}
)
stats3 = analyzer3.get_statistics()

## 7. 여러 작물 비교 분석 (FLDAS)

In [ ]:
# 여러 작물 비교
crops_to_compare = [
    ('wheat', 'wheat'),
    ('rice', 'rice'),
    ('maize', 'maize'),
    ('soybean', 'soybean'),
    ('potato', 'potato'),
]

comparison_results = []

for crop_name, sm_preset in crops_to_compare:
    try:
        analyzer = EcoCropGEE()
        analyzer.load_database('EcoCrop_DB_secondtrim.csv')
        analyzer.set_crop(crop_name)
        analyzer.crop_params.update(get_soil_moisture_preset(sm_preset))
        analyzer.set_roi_point(lon=127.0, lat=36.0, buffer_km=100)
        analyzer.fetch_climate(
            start_date='2020-01-01',
            end_date='2020-12-31',
            source='fldas'
        )
        analyzer.calculate_suitability(
            method='annual',
            weights={'temp': 0.4, 'precip': 0.3, 'soil_moisture': 0.3}
        )
        stats = analyzer.get_statistics(scale=5000)
        
        comparison_results.append({
            'crop': crop_name,
            'overall': stats.get('overall_score_mean', 0),
            'temp': stats.get('temp_score_mean', 0),
            'precip': stats.get('precip_score_mean', 0),
            'soil_moisture': stats.get('soil_moisture_score_mean', 0),
        })
        print(f"✓ {crop_name}: {stats.get('overall_score_mean', 0):.1f}점")
    except Exception as e:
        print(f"✗ {crop_name}: 분석 실패 - {e}")

In [ ]:
# 결과 시각화
if comparison_results:
    df_compare = pd.DataFrame(comparison_results)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = range(len(df_compare))
    width = 0.2
    
    bars1 = ax.bar([i - 1.5*width for i in x], df_compare['temp'], width, label='온도', color='coral')
    bars2 = ax.bar([i - 0.5*width for i in x], df_compare['precip'], width, label='강수량', color='teal')
    bars3 = ax.bar([i + 0.5*width for i in x], df_compare['soil_moisture'], width, label='토양수분', color='saddlebrown')
    bars4 = ax.bar([i + 1.5*width for i in x], df_compare['overall'], width, label='종합', color='steelblue')
    
    ax.set_ylabel('적합성 점수')
    ax.set_title('한국 중부지역 작물별 기후 적합성 비교 (FLDAS 2020)')
    ax.set_xticks(x)
    ax.set_xticklabels(df_compare['crop'])
    ax.legend()
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. 장기 시계열 분석 (1982-2020)

In [ ]:
# 장기 시계열: 연도별 적합성 변화 추적
# (주의: 시간이 오래 걸릴 수 있음)

years_to_analyze = list(range(2015, 2021))  # 2015-2020년
yearly_scores = []

for year in years_to_analyze:
    try:
        analyzer = EcoCropGEE()
        analyzer.load_database('EcoCrop_DB_secondtrim.csv')
        analyzer.set_crop('rice')
        analyzer.crop_params.update(get_soil_moisture_preset('rice'))
        analyzer.set_roi_point(lon=127.0, lat=35.0, buffer_km=50)
        analyzer.fetch_climate(
            start_date=f'{year}-01-01',
            end_date=f'{year}-12-31',
            source='fldas'
        )
        analyzer.calculate_suitability(method='annual')
        stats = analyzer.get_statistics(scale=5000)
        
        yearly_scores.append({
            'year': year,
            'overall': stats.get('overall_score_mean', 0),
            'temp': stats.get('temp_score_mean', 0),
            'precip': stats.get('precip_score_mean', 0),
            'soil_moisture': stats.get('soil_moisture_score_mean', 0),
        })
        print(f"{year}: {stats.get('overall_score_mean', 0):.1f}점")
    except Exception as e:
        print(f"{year}: 분석 실패 - {e}")

In [ ]:
# 연도별 추세 시각화
if yearly_scores:
    df_yearly = pd.DataFrame(yearly_scores)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(df_yearly['year'], df_yearly['overall'], 'o-', label='종합', linewidth=2, markersize=8)
    ax.plot(df_yearly['year'], df_yearly['temp'], 's--', label='온도', alpha=0.7)
    ax.plot(df_yearly['year'], df_yearly['precip'], '^--', label='강수량', alpha=0.7)
    ax.plot(df_yearly['year'], df_yearly['soil_moisture'], 'd--', label='토양수분', alpha=0.7)
    
    ax.set_xlabel('연도')
    ax.set_ylabel('적합성 점수')
    ax.set_title('벼 재배 적합성 연도별 변화 (전라남도)')
    ax.legend()
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 9. 결과 내보내기

In [ ]:
# HTML 지도로 저장
m = analyzer.create_map()
if m:
    m.save('crop_suitability_fldas.html')
    print("지도가 저장되었습니다: crop_suitability_fldas.html")

In [ ]:
# Google Drive로 GeoTIFF 내보내기
# analyzer.export(
#     filename='rice_suitability_fldas_2020',
#     folder='EcoCrop_GEE',
#     scale=1000
# )

---
## 파라미터 참조

### EcoCrop 기본 파라미터
| 파라미터 | 설명 | 단위 |
|----------|------|------|
| TOPMN/TOPMX | 최적 온도 범위 | °C |
| TMIN/TMAX | 허용 온도 범위 | °C |
| ROPMN/ROPMX | 최적 강수량 범위 | mm/년 |
| RMIN/RMAX | 허용 강수량 범위 | mm/년 |

### 토양 수분 파라미터 (FLDAS용)
| 파라미터 | 설명 | 단위 | 일반 범위 |
|----------|------|------|----------|
| SM_MIN | 위조점 (너무 건조) | m³/m³ | 0.10-0.15 |
| SM_OPT1 | 최적 수분 하한 | m³/m³ | 0.18-0.25 |
| SM_OPT2 | 최적 수분 상한 | m³/m³ | 0.32-0.40 |
| SM_MAX | 과습점 (침수 위험) | m³/m³ | 0.40-0.50 |

### 점수 계산 방식
```
제한 요인 방식 (기본):
  종합 점수 = min(온도, 강수량, 토양수분)

가중 평균 방식:
  종합 점수 = 온도*w_t + 강수량*w_p + 토양수분*w_sm
```